In [2]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
import time
import re
import camelot as cam
from PyPDF2 import PdfReader
from datetime import datetime
import pdfplumber
from collections import defaultdict
import logging
import shutil

# Configure logging and start timer
start = time.time()
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
pd.set_option('display.max_rows', 500, 'display.max_columns', 500, 'display.max_colwidth', 1000)

# Setup directories
BASE_DIR = "be_financial_data"
ORIGINAL_DIR = os.path.join(BASE_DIR, "original")
CURRENT_DIR = os.path.join(BASE_DIR, "current")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")

# Create subdirectories for categorized files
for subfolder in ['balance-sheet', 'income-statement']:
    os.makedirs(os.path.join(CURRENT_DIR, subfolder), exist_ok=True)
    print(f"📁 Created directory: {os.path.join(CURRENT_DIR, subfolder)}")

for dir_path in [BASE_DIR, ORIGINAL_DIR, OUTPUT_DIR]:
    os.makedirs(dir_path, exist_ok=True)
    print(f"📁 Created directory: {dir_path}")

# Log file for tracking processed files
PROCESSED_FILES_LOG = os.path.join(BASE_DIR, "processed_files.csv")

bank_url = 'https://bekonomike.com/sq/Bilanci-i-Gjendjes'
print(f"🔗 Target URL: {bank_url}")

def load_processed_files():
    """Load list of already processed files from CSV"""
    if os.path.exists(PROCESSED_FILES_LOG):
        df = pd.read_csv(PROCESSED_FILES_LOG)
        return set(df['file_name'].tolist())
    return set()

def save_processed_file(file_name, status=1, reporting_date=None, current_name=None, is_corrupt=0):
    """Save processed file to log"""
    processed = load_processed_files()
    if file_name not in processed:
        df = pd.DataFrame({
            'file_name': [file_name],
            'processed_date': [datetime.now().strftime('%Y-%m-%d %H:%M:%S')],
            'status': [status],
            'reporting_date': [reporting_date],
            'current_name': [current_name],
            'is_corrupt': [is_corrupt]
        })
        if os.path.exists(PROCESSED_FILES_LOG):
            existing = pd.read_csv(PROCESSED_FILES_LOG)
            df = pd.concat([existing, df], ignore_index=True)
        df.to_csv(PROCESSED_FILES_LOG, index=False)

def scrape_pdf_links(url):
    """Scrape PDF links from the BE page"""
    print(f"🔍 Attempting to connect to: {url}")
    try:
        response = requests.get(url)
        response.raise_for_status()
        print(f"✅ Successfully connected! Status code: {response.status_code}")
        
        soup = BeautifulSoup(response.text, 'html.parser')
        pdf_links = []
        
        for link in soup.find_all('a', href=True):
            href = link['href']
            if href.lower().endswith('.pdf') and not ('manuali' in href.lower() or '303070' in href.lower()):
                full_url = urljoin(url, href)
                pdf_links.append(full_url)
                print(f"   📄 Found PDF: {os.path.basename(full_url)}")
        
        print(f"\n✅ Total PDFs found: {len(pdf_links)}")
        return pdf_links
    except requests.exceptions.RequestException as e:
        print(f"❌ Error connecting to website: {e}")
        return []

def download_pdf(url, local_dir):
    """Download PDF to local directory"""
    file_name = os.path.basename(url)
    local_path = os.path.join(local_dir, file_name)
    
    print(f"   ⬇️ Downloading: {file_name}")
    try:
        response = requests.get(url)
        response.raise_for_status()
        
        with open(local_path, 'wb') as f:
            f.write(response.content)
        
        file_size = os.path.getsize(local_path) / 1024
        print(f"   ✅ Downloaded: {file_name} ({file_size:.1f} KB)")
        return local_path
    except Exception as e:
        print(f"   ❌ Failed to download {file_name}: {e}")
        return None

def extract_date(text, filename):
    """Extract a date string from text or filename"""
    # Try to find date pattern like 31.12.2023 or 31/12/2023
    match = re.search(r'\d{2}[./-]\d{2}[./-]\d{4}', text)
    if match:
        return match.group().replace('-', '.').replace('/', '.')
    
    # Try filename
    match = re.search(r'\d{2}[./-]\d{2}[./-]\d{4}', filename)
    if match:
        return match.group().replace('-', '.').replace('/', '.')
    
    return None

def download_all(force=False):
    """Download PDFs and categorize them locally"""
    processed = set() if force else load_processed_files()
    if force:
        print("⚠️ Force mode enabled: all files will be downloaded")
    
    print(f"🔍 Scraping PDF links from: {bank_url}")
    pdf_links = scrape_pdf_links(bank_url)
    
    if not pdf_links:
        print("❌ No PDF links found")
        return []
    
    new_files = []
    file_metadata = []

    print(f"\n📥 Starting download of {len(pdf_links)} PDFs...")
    
    for i, pdf_url in enumerate(pdf_links, 1):
        file_name = os.path.basename(pdf_url)
        print(f"\n[{i}/{len(pdf_links)}] Processing: {file_name}")
        
        # Skip if already processed
        if file_name in processed and not force:
            print(f"   ⏭️ File already processed. Skipping...")
            continue

        new_files.append(file_name)
        
        # Download to original directory
        file_path = download_pdf(pdf_url, ORIGINAL_DIR)
        if not file_path:
            continue

        # Try to read metadata creation date
        dt = None
        try:
            with open(file_path, 'rb') as f:
                pdf_reader = PdfReader(f)
                metadata = pdf_reader.metadata
                if metadata:
                    for k, v in metadata.items():
                        if "/CreationDate" in k:
                            date_str = re.search(r'D:(\d+)', str(v))
                            if date_str:
                                dt = datetime.strptime(date_str.group(1), '%Y%m%d%H%M%S')
                                print(f"   📅 PDF creation date: {dt.date()}")
                                break
        except Exception as e:
            print(f"   ⚠️ Could not read metadata: {e}")

        # Extract date from PDF content
        date = None
        try:
            with pdfplumber.open(file_path) as pdf:
                full_text = "".join(page.extract_text() or "" for page in pdf.pages[:3])
                date = extract_date(full_text, file_name)
        except Exception as e:
            print(f"   ⚠️ Error extracting text: {e}")

        if date:
            print(f"   📅 Extracted reporting date: {date}")
        else:
            print(f"   ⚠️ Could not extract date")
            date = datetime.now().strftime('%d.%m.%Y')  # Use current date as fallback

        # For BE, we need to create both BS and IS from the same PDF
        bs_new_name = f'be_bs_{date}.pdf'
        is_new_name = f'be_is_{date}.pdf'
        
        # Copy to both categorized folders
        shutil.copy2(file_path, os.path.join(CURRENT_DIR, 'balance-sheet', bs_new_name))
        shutil.copy2(file_path, os.path.join(CURRENT_DIR, 'income-statement', is_new_name))
        
        file_metadata.append({
            'file_name': file_name,
            'file_path': file_path,
            'bs_current_name': bs_new_name,
            'is_current_name': is_new_name,
            'reporting_date': date,
            'creation_date': dt,
            'download_date': datetime.now(),
            'is_corrupt': 0
        })
        
        print(f"   ✅ Created: {bs_new_name} and {is_new_name}")
        
        # Mark as processed (store both names combined)
        save_processed_file(file_name, reporting_date=date, current_name=f"{bs_new_name}|{is_new_name}")

    # Save metadata
    if file_metadata:
        metadata_df = pd.DataFrame(file_metadata)
        metadata_path = os.path.join(OUTPUT_DIR, f"file_metadata_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv")
        metadata_df.to_csv(metadata_path, index=False)
        print(f"\n💾 File metadata saved to {metadata_path}")

    if new_files:
        print(f"\n✅ Downloaded {len(new_files)} new files.")
    else:
        print("\n✅ No new files detected.")

    # Return list of all new current files
    current_files = []
    for m in file_metadata:
        current_files.append(m['bs_current_name'])
        current_files.append(m['is_current_name'])
    return current_files

def replace_negatives(value):
    """Convert parenthesized numbers to negative"""
    if pd.isna(value) or str(value).strip() in ['', '-', '0']:
        return '0'
    val = str(value).strip()
    if '(' in val and ')' in val:
        num = val.replace('(', '').replace(')', '').replace(',', '').strip()
        return f"-{num}"
    return val

def read_be_pdf(pdf_path, pdf_filename):
    """Extract data from BE PDF using camelot"""
    try:
        print(f"      📊 Reading PDF with camelot: {pdf_filename}")
        tables = cam.read_pdf(pdf_path, flavor='stream')
        dfs = []
        for table in tables:
            df = table.df
            dfs.append(df)
        
        full_df = pd.concat(dfs, ignore_index=True)
        
        # Split into balance sheet and income statement (BE has both in same PDF)
        balance_df = full_df[[0, 1, 2]].copy()
        income_df = full_df[[3, 4, 5]].copy()
        
        # Clean balance sheet
        balance_df.drop([0, 1, 2, 3, 4], inplace=True, errors='ignore')
        balance_df.dropna(subset=[1, 2], inplace=True)
        balance_df = balance_df[(balance_df[1] != '') & (balance_df[2] != '')]
        balance_df = balance_df[(balance_df[1].astype(str).str.strip() != '') & (balance_df[2].astype(str).str.strip() != '')]
        
        # Clean income statement
        income_df.drop([0, 1, 2, 3, 4], inplace=True, errors='ignore')
        income_df.dropna(subset=[4, 5], inplace=True)
        income_df = income_df[(income_df[4] != '') & (income_df[5] != '')]
        income_df = income_df[(income_df[4].astype(str).str.strip() != '') & (income_df[5].astype(str).str.strip() != '')]
        
        # Set column names
        income_df.columns = ['INCOME_CATEGORY', 'PREVIOUS_INCOME_VALUE', 'CURRENT_INCOME_VALUE']
        balance_df.columns = ['BALANCE_CATEGORY', 'PREVIOUS_BALANCE_VALUE', 'CURRENT_BALANCE_VALUE']
        
        # Add file name
        income_df['FILE_NAME'] = pdf_filename
        balance_df['FILE_NAME'] = pdf_filename
        
        # Clean category names
        balance_df['BALANCE_CATEGORY'] = balance_df['BALANCE_CATEGORY'].str.replace('\n', ' ').str.strip()
        income_df['INCOME_CATEGORY'] = income_df['INCOME_CATEGORY'].str.replace('\n', ' ').str.strip()
        
        print(f"      ✅ Extracted: {len(balance_df)} balance rows, {len(income_df)} income rows")
        return income_df, balance_df
    
    except Exception as e:
        print(f"      ❌ Error reading PDF with camelot: {e}")
        return pd.DataFrame(), pd.DataFrame()

# Data storage
income_data = defaultdict(list)
balance_data = defaultdict(list)

def process_be_files():
    """Process all BE files and populate data dictionaries"""
    
    # Process balance sheet files
    bs_dir = os.path.join(CURRENT_DIR, 'balance-sheet')
    if os.path.exists(bs_dir):
        print(f"\n📂 Processing balance sheet files...")
        for filename in os.listdir(bs_dir):
            if filename.endswith('.pdf') and filename.startswith('be_bs_'):
                file_path = os.path.join(bs_dir, filename)
                print(f"\n   🔍 Processing: {filename}")
                
                # Extract date from filename
                date_match = re.search(r'\d{2}\.\d{2}\.\d{4}', filename)
                dt_report = date_match.group() if date_match else None
                
                if not dt_report:
                    print(f"      ⚠️ Could not extract date from filename")
                    continue
                
                # Get the corresponding income statement filename
                is_filename = filename.replace('bs', 'is')
                is_path = os.path.join(CURRENT_DIR, 'income-statement', is_filename)
                
                if os.path.exists(is_path):
                    # Read the PDF (use either file, they're the same content)
                    income_df, balance_df = read_be_pdf(file_path, filename)
                    
                    if not balance_df.empty and dt_report:
                        for _, row in balance_df.iterrows():
                            balance_data[dt_report].append({
                                'Category': row['BALANCE_CATEGORY'],
                                'PREVIOUS_QUARTER': row['PREVIOUS_BALANCE_VALUE'],
                                'CURRENT_QUARTER': row['CURRENT_BALANCE_VALUE'],
                                'DT_REPORT': dt_report,
                                'FILE_NAME': filename
                            })
                        print(f"      ✅ Added {len(balance_df)} balance rows for {dt_report}")
                    
                    if not income_df.empty and dt_report:
                        for _, row in income_df.iterrows():
                            income_data[dt_report].append({
                                'Category': row['INCOME_CATEGORY'],
                                'PREVIOUS_QUARTER': row['PREVIOUS_INCOME_VALUE'],
                                'CURRENT_QUARTER': row['CURRENT_INCOME_VALUE'],
                                'DT_REPORT': dt_report,
                                'FILE_NAME': is_filename
                            })
                        print(f"      ✅ Added {len(income_df)} income rows for {dt_report}")

def clean_numeric_values(df, prev_col, curr_col):
    """Clean and convert numeric columns"""
    for col in [prev_col, curr_col]:
        df[col] = df[col].astype(str).str.replace('-', '0').apply(replace_negatives)
        df[col] = pd.to_numeric(df[col].str.replace(',', ''), errors='coerce').fillna(0)
    return df

def main(force=False, extract_only=False):
    print("\n" + "="*60)
    print("🏦 BE BANK (Banka Ekonomike) FINANCIAL DATA EXTRACTION")
    print("="*60)
    
    global income_data, balance_data
    income_data.clear()
    balance_data.clear()
    
    # Get files to process
    if extract_only:
        print("\n📂 EXTRACT-ONLY MODE: Processing existing files")
        new_files = []
        for category in ['balance-sheet', 'income-statement']:
            cat_dir = os.path.join(CURRENT_DIR, category)
            if os.path.exists(cat_dir):
                category_files = [f for f in os.listdir(cat_dir) if f.endswith('.pdf')]
                new_files.extend(category_files)
        print(f"Found {len(new_files)} existing files")
    else:
        print("\n🌐 DOWNLOAD MODE")
        new_files = download_all(force=force)
    
    if new_files or extract_only:
        # Process all BE files
        process_be_files()
        
        # Create DataFrames
        print("\n📊 Creating DataFrames...")
        
        # Balance Sheet DataFrame
        full_bs = pd.DataFrame()
        for date, data in balance_data.items():
            df = pd.DataFrame(data)
            full_bs = pd.concat([full_bs, df], ignore_index=True)
        
        if not full_bs.empty:
            full_bs.columns = ['BALANCE_CATEGORY', 'PREVIOUS_BALANCE_VALUE', 'CURRENT_BALANCE_VALUE', 'DT_REPORT', 'FILE_NAME']
            full_bs = clean_numeric_values(full_bs, 'PREVIOUS_BALANCE_VALUE', 'CURRENT_BALANCE_VALUE')
            
            # Category mapping for balance sheet
            bs_map = {
                'Paraja e gatshme dhe gjendja me BQK-në': '20',
                'Kërkesat ndaj bankave': '21',
                'Bonot e thesarit': '22',
                'Investimet në letra me vlerë': '23',
                'Kreditë dhe paradhëniet ndaj klientëve': '26',
                'Patundshmëritë dhe pajisjet': '27',
                'Pasuritë e paprekshme': '28',
                'Pasuritë tatimore të shtyra': '29',
                'Pasuritë tjera': '30',
                'Gjithsej pasuritë': '31',
                'Depozitat e klientëve': '32',
                'Detyrime ndaj klientëve': '32',
                'Detyrime ndaj bankave': '33',
                'Detyrimet ndaj bankave': '33',
                'Fondet tjera të huazuara': '34',
                'Detyrimet tatimore të shtyra': '35',
                'Detyrimet tjera': '36',
                'Gjithsej detyrimet': '37',
                'Kapitali aksionar': '38',
                'Rezervat e kapitalit': '40',
                'Fitimi i mbajtur/(humbja) nga vitet paraprake': '41',
                'Fitimi i mbajtur /(humbja) nga vitet paraprake': '41',
                'Fitimi/(humbja) e vitit aktual': '42',
                'Përbërësit tjerë të ekuitetit': '43',
                'Gjithsej ekuiteti i aksionarëve': '44',
                'Gjithsej  ekuiteti i aksionarëve': '44',
                'Gjithsej detyrimet dhe ekuiteti i aksionarëve': '45',
                'Gjithsej  detyrimet  dhe ekuiteti i aksionarëve': '45'
            }
            
            full_bs['CATEGORY_CODE'] = full_bs['BALANCE_CATEGORY'].map(bs_map)
            full_bs['BANK_ID'] = 3  # BE Bank ID
            full_bs['STATEMENT_TYPE'] = 'BALANCE_SHEET'
            full_bs['DT_REPORT'] = pd.to_datetime(full_bs['DT_REPORT'], format='%d.%m.%Y', errors='coerce')
        
        # Income Statement DataFrame
        full_is = pd.DataFrame()
        for date, data in income_data.items():
            df = pd.DataFrame(data)
            full_is = pd.concat([full_is, df], ignore_index=True)
        
        if not full_is.empty:
            full_is.columns = ['INCOME_CATEGORY', 'PREVIOUS_INCOME_VALUE', 'CURRENT_INCOME_VALUE', 'DT_REPORT', 'FILE_NAME']
            full_is = clean_numeric_values(full_is, 'PREVIOUS_INCOME_VALUE', 'CURRENT_INCOME_VALUE')
            
            # Category mapping for income statement
            is_map = {
                'Të hyrat nga interesi': '1',
                'Shpenzimet e interesit': '2',
                'Neto të hyrat nga interesi': '3',
                'Të hyrat nga tarifat dhe komisionet': '4',
                'Shpenzimet e tarifave dhe komisioneve': '5',
                'Neto të hyrat nga tarifat dhe komisionet': '6',
                'Neto të hyrat nga tregtimi': '7',
                'Neto të hyrat nga instrumentet tjera financiare': '8',
                'Neto të hyrat (shpenzimet) tjera operative': '9',
                'Neto të hyrat (shpenzimet)  tjera operative': '9',
                'Gjithsej të hyrat': '10',
                'Gjithsej  të hyrat': '10',
                'Provizionet për humbjet nga kreditë': '11',
                'Fitimi/(humbja) para tatimit': '14',
                'Shpenzimet e tatimit në fitim': '15',
                'Fitimi/(humbja) neto': '16',
                'Të ardhurat tjera gjithëpërfshirëse': '17',
                'Gjithsej të ardhurat gjithëpërfshirëse': '19',
                'Gjithsej  të ardhurat gjithëpërfshirëse': '19',
                'Interest income': '1',
                'Interest expenses': '2',
                'Net interest income': '3',
                'Fee and commission income': '4',
                'Fee and commission expenses': '5',
                'Net fee and commission income': '6',
                'Foreign exchange gains/(losses)': '7',
                'Net gains from other financial instruments': '8',
                'Neto other operating income (expenses)': '9',
                'Total Income': '10',
                'Loan loss provision charges': '11',
                'Net profit before taxes': '14',
                'Income tax expenses': '15',
                'Net profit for the period': '16',
                'Other comprehensive income': '17',
                'Total other comprehensive income': '19'
            }
            
            full_is['CATEGORY_CODE'] = full_is['INCOME_CATEGORY'].map(is_map)
            full_is['BANK_ID'] = 3  # BE Bank ID
            full_is['STATEMENT_TYPE'] = 'INCOME_STATEMENT'
            full_is['DT_REPORT'] = pd.to_datetime(full_is['DT_REPORT'], format='%d.%m.%Y', errors='coerce')
        
        # Create unified DataFrame
        unified_dfs = []
        
        if not full_is.empty:
            is_unified = full_is.rename(columns={
                'INCOME_CATEGORY': 'ORIGINAL_CATEGORY',
                'PREVIOUS_INCOME_VALUE': 'PREVIOUS_VALUE',
                'CURRENT_INCOME_VALUE': 'CURRENT_VALUE'
            })
            is_unified['CATEGORY_TYPE'] = 'INCOME'
            unified_dfs.append(is_unified)
        
        if not full_bs.empty:
            bs_unified = full_bs.rename(columns={
                'BALANCE_CATEGORY': 'ORIGINAL_CATEGORY',
                'PREVIOUS_BALANCE_VALUE': 'PREVIOUS_VALUE',
                'CURRENT_BALANCE_VALUE': 'CURRENT_VALUE'
            })
            bs_unified['CATEGORY_TYPE'] = 'BALANCE'
            unified_dfs.append(bs_unified)
        
        if unified_dfs:
            final_df = pd.concat(unified_dfs, ignore_index=True)
            final_df['BANK_ID'] = 3
            final_df['CURR_ID'] = 1
            final_df['DATA_SOURCE'] = 'Banka Ekonomike'
            final_df['EXTRACTION_DATE'] = datetime.now().strftime('%Y-%m-%d')
            final_df['REPORT_DATE'] = final_df['DT_REPORT'].dt.strftime('%Y-%m-%d')
            
            # Drop rows with invalid dates
            final_df = final_df.dropna(subset=['DT_REPORT'])
            
            # Sort
            final_df = final_df.sort_values(['DT_REPORT', 'STATEMENT_TYPE', 'CATEGORY_CODE'])
            
            # Reorder columns
            column_order = ['BANK_ID', 'REPORT_DATE', 'DT_REPORT', 'STATEMENT_TYPE', 'CATEGORY_TYPE',
                          'CATEGORY_CODE', 'ORIGINAL_CATEGORY', 'PREVIOUS_VALUE', 'CURRENT_VALUE',
                          'CURR_ID', 'FILE_NAME', 'DATA_SOURCE', 'EXTRACTION_DATE']
            
            final_columns = [col for col in column_order if col in final_df.columns]
            final_df = final_df[final_columns]
            
            # Save outputs
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            
            # Save unified DataFrame
            unified_path = os.path.join(OUTPUT_DIR, f"be_unified_financial_data_{timestamp}.csv")
            final_df.to_csv(unified_path, index=False)
            print(f"\n💾 Unified data saved to: {unified_path}")
            
            # Save individual files
            if not full_is.empty:
                is_path = os.path.join(OUTPUT_DIR, f"be_income_statement_{timestamp}.csv")
                full_is.to_csv(is_path, index=False)
            
            if not full_bs.empty:
                bs_path = os.path.join(OUTPUT_DIR, f"be_balance_sheet_{timestamp}.csv")
                full_bs.to_csv(bs_path, index=False)
            
            # Save Excel with multiple sheets
            excel_path = os.path.join(OUTPUT_DIR, f"be_financial_report_{timestamp}.xlsx")
            with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
                final_df.to_excel(writer, sheet_name='Unified_Data', index=False)
                if not full_is.empty:
                    full_is.to_excel(writer, sheet_name='Income_Statement', index=False)
                if not full_bs.empty:
                    full_bs.to_excel(writer, sheet_name='Balance_Sheet', index=False)
            
            print(f"💾 Excel report saved to: {excel_path}")
            
            print(f"\n📊 UNIFIED DATAFRAME CREATED:")
            print(f"   - Total rows: {len(final_df)}")
            print(f"   - Income Statement rows: {len(final_df[final_df['STATEMENT_TYPE'] == 'INCOME_STATEMENT'])}")
            print(f"   - Balance Sheet rows: {len(final_df[final_df['STATEMENT_TYPE'] == 'BALANCE_SHEET'])}")
            print(f"   - Unique report dates: {final_df['REPORT_DATE'].nunique()}")
            print(f"   - Date range: {final_df['REPORT_DATE'].min()} to {final_df['REPORT_DATE'].max()}")
            
            elapsed_time = time.time() - start
            print(f"\n⏱️ Processing completed in {elapsed_time:.2f} seconds")
            
            return final_df, full_is, full_bs
        
        return pd.DataFrame(), full_is, full_bs
    
    return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

# Run the extraction
print("🚀 Starting BE Bank financial data extraction...")
final_df, income_df, balance_df = main(force=True, extract_only=False)

if not final_df.empty:
    print("\n" + "="*60)
    print("📊 FINAL UNIFIED DATAFRAME")
    print("="*60)
    print(f"Shape: {final_df.shape}")
    print("\nFirst 10 rows:")
    print(final_df.head(10))
    
    print("\n📊 Summary by Statement Type:")
    print(final_df['STATEMENT_TYPE'].value_counts())
    
    print("\n📅 Reports by Date:")
    print(final_df['REPORT_DATE'].value_counts().sort_index())
else:
    print("❌ No data in final DataFrame")

📁 Created directory: be_financial_data/current/balance-sheet
📁 Created directory: be_financial_data/current/income-statement
📁 Created directory: be_financial_data
📁 Created directory: be_financial_data/original
📁 Created directory: be_financial_data/output
🔗 Target URL: https://bekonomike.com/sq/Bilanci-i-Gjendjes
🚀 Starting BE Bank financial data extraction...

🏦 BE BANK (Banka Ekonomike) FINANCIAL DATA EXTRACTION

🌐 DOWNLOAD MODE
⚠️ Force mode enabled: all files will be downloaded
🔍 Scraping PDF links from: https://bekonomike.com/sq/Bilanci-i-Gjendjes
🔍 Attempting to connect to: https://bekonomike.com/sq/Bilanci-i-Gjendjes
✅ Successfully connected! Status code: 200
   📄 Found PDF: 508089_gjendja_e_bilancit_(shqip).pdf
   📄 Found PDF: 122462_gjendja_e_bilancit_(shqip).pdf
   📄 Found PDF: 328344_gjendja_e_bilancit_(shqip).pdf
   📄 Found PDF: 271566_gjendja_e_bilancit_03_2025_(3).pdf
   📄 Found PDF: 319454_gjendja_e_bilancit_12_2024.pdf
   📄 Found PDF: 852402_PUBLIKIMI_I_BILANCIT_SHQIP

2026-03-15T18:50:42 - INFO - Processing page-1
2026-03-15 18:50:42,542 - INFO - Processing page-1


   ✅ Downloaded: 165565_Bilanci_i_tremujorit_te_trete_te_2009.pdf (1187.7 KB)
   📅 PDF creation date: 2015-11-04
   ⚠️ Could not extract date
   ✅ Created: be_bs_15.03.2026.pdf and be_is_15.03.2026.pdf

💾 File metadata saved to be_financial_data/output/file_metadata_20260315_185042.csv

✅ Downloaded 66 new files.

📂 Processing balance sheet files...

   🔍 Processing: be_bs_31.03.2024.pdf
      📊 Reading PDF with camelot: be_bs_31.03.2024.pdf


2026-03-15T18:50:42 - INFO - Processing page-1
2026-03-15 18:50:42,661 - INFO - Processing page-1
2026-03-15T18:50:42 - INFO - Processing page-1
2026-03-15 18:50:42,786 - INFO - Processing page-1


      ✅ Extracted: 23 balance rows, 16 income rows
      ✅ Added 23 balance rows for 31.03.2024
      ✅ Added 16 income rows for 31.03.2024

   🔍 Processing: be_bs_31.03.2018.pdf
      📊 Reading PDF with camelot: be_bs_31.03.2018.pdf
      ✅ Extracted: 24 balance rows, 17 income rows
      ✅ Added 24 balance rows for 31.03.2018
      ✅ Added 17 income rows for 31.03.2018

   🔍 Processing: be_bs_31.03.2019.pdf
      📊 Reading PDF with camelot: be_bs_31.03.2019.pdf


...

      ✅ Extracted: 22 balance rows, 15 income rows
      ✅ Added 22 balance rows for 31.03.2014
      ✅ Added 15 income rows for 31.03.2014

   🔍 Processing: be_bs_31.03.2015.pdf
      📊 Reading PDF with camelot: be_bs_31.03.2015.pdf


2026-03-15T18:50:51 - INFO - Processing page-1
2026-03-15 18:50:51,488 - INFO - Processing page-1


      ✅ Extracted: 22 balance rows, 16 income rows
      ✅ Added 22 balance rows for 31.03.2015
      ✅ Added 16 income rows for 31.03.2015

   🔍 Processing: be_bs_31.03.2017.pdf
      📊 Reading PDF with camelot: be_bs_31.03.2017.pdf
      ✅ Extracted: 24 balance rows, 17 income rows
      ✅ Added 24 balance rows for 31.03.2017
      ✅ Added 17 income rows for 31.03.2017

   🔍 Processing: be_bs_31.03.2016.pdf
      📊 Reading PDF with camelot: be_bs_31.03.2016.pdf


2026-03-15T18:50:51 - INFO - Processing page-1
2026-03-15 18:50:51,621 - INFO - Processing page-1


      ✅ Extracted: 23 balance rows, 17 income rows
      ✅ Added 23 balance rows for 31.03.2016
      ✅ Added 17 income rows for 31.03.2016

📊 Creating DataFrames...

💾 Unified data saved to: be_financial_data/output/be_unified_financial_data_20260315_185051.csv
💾 Excel report saved to: be_financial_data/output/be_financial_report_20260315_185051.xlsx

📊 UNIFIED DATAFRAME CREATED:
   - Total rows: 2066
   - Income Statement rows: 853
   - Balance Sheet rows: 1213
   - Unique report dates: 52
   - Date range: 2012-12-31 to 2025-12-31

⏱️ Processing completed in 30.90 seconds

📊 FINAL UNIFIED DATAFRAME
Shape: (2066, 13)

First 10 rows:
      BANK_ID REPORT_DATE  DT_REPORT STATEMENT_TYPE CATEGORY_TYPE  \
1583        3  2012-12-31 2012-12-31  BALANCE_SHEET       BALANCE   
1584        3  2012-12-31 2012-12-31  BALANCE_SHEET       BALANCE   
1585        3  2012-12-31 2012-12-31  BALANCE_SHEET       BALANCE   
1586        3  2012-12-31 2012-12-31  BALANCE_SHEET       BALANCE   
1587        3

In [4]:
final_df[final_df['REPORT_DATE'] == '2025-12-31']

,BANK_ID,REPORT_DATE,DT_REPORT,STATEMENT_TYPE,CATEGORY_TYPE,CATEGORY_CODE,ORIGINAL_CATEGORY,PREVIOUS_VALUE,CURRENT_VALUE,CURR_ID,FILE_NAME,DATA_SOURCE,EXTRACTION_DATE
1230,3,2025-12-31,2025-12-31,BALANCE_SHEET,BALANCE,20,Paraja e gatshme dhe gjendja me BQK-në,65059.0,86432.0,1,be_bs_31.12.2025.pdf,Banka Ekonomike,2026-03-15
1231,3,2025-12-31,2025-12-31,BALANCE_SHEET,BALANCE,21,Kërkesat ndaj bankave,39203.0,26191.0,1,be_bs_31.12.2025.pdf,Banka Ekonomike,2026-03-15
1232,3,2025-12-31,2025-12-31,BALANCE_SHEET,BALANCE,22,Bonot e thesarit,0.0,0.0,1,be_bs_31.12.2025.pdf,Banka Ekonomike,2026-03-15
1233,3,2025-12-31,2025-12-31,BALANCE_SHEET,BALANCE,23,Investimet në letra me vlerë,96871.0,99952.0,1,be_bs_31.12.2025.pdf,Banka Ekonomike,2026-03-15
1234,3,2025-12-31,2025-12-31,BALANCE_SHEET,BALANCE,26,Kreditë dhe paradhëniet ndaj klientëve,552014.0,561627.0,1,be_bs_31.12.2025.pdf,Banka Ekonomike,2026-03-15
1235,3,2025-12-31,2025-12-31,BALANCE_SHEET,BALANCE,27,Patundshmëritë dhe pajisjet,14106.0,14685.0,1,be_bs_31.12.2025.pdf,Banka Ekonomike,2026-03-15
1236,3,2025-12-31,2025-12-31,BALANCE_SHEET,BALANCE,28,Pasuritë e paprekshme,2279.0,2174.0,1,be_bs_31.12.2025.pdf,Banka Ekonomike,2026-03-15
1237,3,2025-12-31,2025-12-31,BALANCE_SHEET,BALANCE,29,Pasuritë tatimore të shtyra,0.0,0.0,1,be_bs_31.12.2025.pdf,Banka Ekonomike,2026-03-15
1238,3,2025-12-31,2025-12-31,BALANCE_SHEET,BALANCE,30,Pasuritë tjera,21631.0,8957.0,1,be_bs_31.12.2025.pdf,Banka Ekonomike,2026-03-15
1239,3,2025-12-31,2025-12-31,BALANCE_SHEET,BALANCE,31,Gjithsej pasuritë,791162.0,800019.0,1,be_bs_31.12.2025.pdf,Banka Ekonomike,2026-03-15
